In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
from fastai.vision.all import *
from fastai.callback.hook import *
from fastai.test_utils import *
from glob import glob
import skimage.io
from PIL import Image
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import random

# Training

In [ ]:
path_annotations = Path('./hjk_nucleolin_40x_ESC_E14.5_combined/annotations/')
path_images = Path('./hjk_nucleolin_40x_ESC_E14.5_combined/images/')

In [ ]:
codes = np.array(['Background','Nucleolus'], dtype='<U17')

In [ ]:
fnames = get_image_files(Path('./hjk_nucleolin_40x_ESC_E14.5_combined/images/all'))
def label_func(x): return x.parent.parent.parent/'annotations'/(x.stem+'.png')

In [ ]:
label_func(fnames[0])

In [ ]:
src = SegmentationDataLoaders.from_label_func(path_images, fnames, label_func, classes=codes,
                                              item_tfms=Resize((224,224)),
                                             batch_tfms=[*aug_transforms(flip_vert=True), Resize(ResizeMethod.Pad), Normalize.from_stats(*imagenet_stats)])
src

In [ ]:
src.vocab = codes

In [ ]:
src.show_batch()

In [ ]:
bs=16

In [ ]:
class Precision_Recall_Segmentation(Callback):   
    _order=-20 # Needs to run before the recorder
    def __init__(self, learn):
        super().__init__(learn)
    
    def on_train_begin(self, **kwargs):
        self.learn.recorder.add_metric_names(['precision', 'recall'])
    
    def on_epoch_begin(self, **kwargs):
        self.tp = 0
        self.fp = 0
        self.tn = 0
        self.fn = 0
    
    def on_batch_end(self, last_output, last_target, **kwargs):
        #get ious
        iou_cutoff=0.01
        
        #get shapes and flatten
        n = last_target.shape[0]
        last_output = last_output.argmax(dim=1).view(n,-1)
        last_target = last_target.view(n,-1)
        
        #get ious
        ious = torch.zeros([n])
        input_ = (last_output==1)
        targs_ = (last_target==1)
        intersect_ = (input_*targs_).sum(dim = 1).float()
        union_ = (input_+targs_).sum(dim = 1).float()
        ious = intersect_ / (union_-intersect_+1.0)
        
        pred_pos = input_.any(1, keepdim=False)
        gt_pos = targs_.any(1, keepdim=False)
        ious = (ious >= iou_cutoff)
        
        #compute tp, fp, tn, fn
        self.tp += (pred_pos * (gt_pos * ious)).sum()
        self.fp += ((pred_pos)*(~(gt_pos*ious))).sum()
        self.tn += (~((pred_pos | gt_pos) | ious)).sum()
        self.fn += (((~pred_pos)*gt_pos)*(~ious)).sum()
        
    
    def on_epoch_end(self, last_metrics, **kwargs):
        return add_metrics(last_metrics, [self.tp.float()/((self.tp+self.fp).float()), 
                                          self.tp.float()/((self.tp+self.fn).float())])

In [ ]:
wd=1e-2

In [ ]:
opt = ranger
learn34 = unet_learner(src, models.resnet34, wd=wd, opt_func=opt, self_attention=False, act_cls=Mish)


In [ ]:
learn34.lr_find()

In [ ]:
learn34.recorder.plot_lr_find()

In [ ]:
lr=1e-4

In [ ]:
learn34.fit_one_cycle(5, slice(lr/10,lr), pct_start=0.9)

In [ ]:
learn34.show_results(figsize=(10,150))

In [ ]:
learn34.fit_one_cycle(5, slice(lr/10,lr), pct_start=0.9)

In [ ]:
lr=1e-5
learn34.fit_one_cycle(5, slice(lr/10,lr), pct_start=0.9)

In [ ]:
learn34.unfreeze()

In [ ]:
learn34.lr_find()

In [ ]:
lrs = slice(5e-6,5e-5)

In [ ]:
learn34.fit_one_cycle(5,lrs,wd=1e-3)

In [ ]:
learn34.fit_one_cycle(10,lrs,wd=1e-3)

In [ ]:
lrs = slice(1e-6,5e-5)
learn34.fit_one_cycle(30,lrs,wd=1e-3)

In [ ]:
learn34.show_results(max_n=100, shuffle=False, )

In [ ]:
learn34.save('resnet34-unet-nucleolus-40xdry-combined-ESC-E14.5')

In [ ]:
learn34.export()

## Check performance

In [ ]:
def process_annotation(path, number_of_object_classes, length_of_object_classes, labels):
    
    annotation = skimage.io.imread(path)
    if len(annotation.shape) == 3:
        annotation = annotation[:,:,0]
    
    cumsum = 1
    groupings = [[] for i in range(number_of_object_classes)]
    for i in range(number_of_object_classes):
        groupings[i] = list(range(cumsum,cumsum+length_of_object_classes[i]))
        cumsum = cumsum + length_of_object_classes[i]

    masks = {}
    
    for i in range(number_of_object_classes):
        grouping = groupings[i]
        masks[labels[i]] = []
        for value in grouping:
            differentiated_masks = skimage.measure.label(annotation == value)
            n_masks = np.max(differentiated_masks)
            for j in range(1,n_masks+1):
                mask = np.asarray((differentiated_masks==j),dtype='uint8')
                mask = scipy.ndimage.morphology.binary_closing(mask)
                mask = 255*scipy.ndimage.morphology.binary_fill_holes(mask)
                masks[labels[i]].append(mask)
                
    return masks

number_of_object_classes = 3
length_of_object_classes = [5,3,1]
labels = ['nucleolus','outoffocus','nosignal']
if len(labels) is not number_of_object_classes:
    print('Not the correct number of labels!')

In [ ]:
test_crop_dir = '/home/hyeonjin/projects/annotation_processing/hjk_nucleolin_40xdry_v1/hjk_nucleolin_40x_ESC_E14.5_combined/images/test'
gt_anno_dir = '/home/hyeonjin/projects/annotation_processing/hjk_nucleolin_40xdry_v1/hjk_nucleolin_40x_ESC_E14.5_combined/annotations'
test_crop_files = glob(os.path.join(test_crop_dir, '*.png'))

list_of_test_crops = []
list_of_gt_crops = []

for file in test_crop_files:
    list_of_test_crops.append(skimage.io.imread(file))
    filename = os.path.basename(file)
    list_of_gt_crops.append(skimage.io.imread(os.path.join(gt_anno_dir, filename)))
    


In [ ]:
segmentation_model_path = '/home/hyeonjin/Projects/fastai_models/compartment_segmentation/export.pkl'

batch_size = 8

n_cells = len(list_of_test_crops)    

def label_func(x): return x.parent.parent.parent/'annotations'/(x.stem+'.png')
learn = load_learner(segmentation_model_path, cpu=False)

test_dl = learn.dls.test_dl(list_of_test_crops)
preds = torch.argmax(learn.get_preds(dl = test_dl)[0], axis=1)


In [ ]:
item_tfms=Resize((224,224)),
batch_tfms=[*aug_transforms(flip_vert=True), Resize(ResizeMethod.Pad), Normalize.from_stats(*imagenet_stats)]

In [ ]:
ii = 100

for i in range(ii):
    fig = plt.figure(figsize=(8,8))

    ax = fig.add_subplot(1,3,1)
    ax.imshow(list_of_test_crops[i])
    ax.axis('off')
    
    ax = fig.add_subplot(1,3,2)
    ax.imshow(preds[i].numpy())
    ax.axis('off')
    
    ax = fig.add_subplot(1,3,3)
    ax.imshow(list_of_gt_crops[i])
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
jaccard_indices = []
dice_coefficients = []
area_prop = []

for i in range(len(list_of_test_crops)):
    
    pred_1 = (preds[i].numpy() > 0.5).astype(np.uint8)
    gt_1 = (cv2.resize(list_of_gt_crops[i], (224, 224)) > 0.5).astype(np.uint8)
    
    intersection = np.logical_and(pred_1, gt_1).sum()
    union = np.logical_or(pred_1, gt_1).sum()
    pred_area = pred_1.sum()
    gt_area = gt_1.sum()

    jaccard = intersection / union if union > 0 else 0
    jaccard_indices.append(jaccard)

    dice = (2 * intersection) / (pred_area + gt_area) if (pred_area + gt_area) > 0 else 0
    dice_coefficients.append(dice)

    prop_area = pred_area / gt_area if gt_area > 0 else 0
    area_prop.append(prop_area)


print(f"--- Results for {len(jaccard_indices)} Speckle Crops ---")
print(f"Median Jaccard (IoU): {np.median(jaccard_indices):.4f}")
print(f"Median Dice:           {np.median(dice_coefficients):.4f}")
print(f"Median Area Prop:      {np.median(area_prop):.4f}")

In [ ]:
data = [jaccard_indices, dice_coefficients]
n = len(jaccard_indices)
print(f"--- Results for {n} Nucleolus Crops ---")
print(f"Median Jaccard (IoU): {np.median(jaccard_indices):.4f}")
print(f"Median Dice:           {np.median(dice_coefficients):.4f}")


plt.figure(figsize=(4, 5), dpi=300)

bp = plt.boxplot(data, labels=['Jaccard Index\n(IoU)', 'Dice\nCoefficient'], 
                 patch_artist=True, widths=0.6)

colors = ['#1f77b4', '#ff7f0e']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for median in bp['medians']:
    median.set(color='black', linewidth=1)

plt.ylabel('Score', fontsize=14)
plt.title(f'Nucleolar segmentation (n={n} crops)', fontsize=14, pad=15)
plt.ylim(0.35, 1.05)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## Speckles

In [ ]:
test_crop_dir = '/home/hyeonjin/trapnell-learning/Projects/4DN_embryo_imaging/speckle_crops_only/'
gt_anno_dir = '/home/hyeonjin/trapnell-learning/Projects/4DN_embryo_imaging/speckle_crops_only_annotations/'
test_crop_files = glob(os.path.join(gt_anno_dir, '*.png'))

list_of_test_crops = []
list_of_gt_crops = []

for file in test_crop_files:
    list_of_gt_crops.append(skimage.io.imread(file)[:,:,0])
    filename = os.path.basename(file)
    list_of_test_crops.append(skimage.io.imread(os.path.join(test_crop_dir, filename))[:,:,0])
    


In [ ]:
segmentation_model_path = '/home/hyeonjin/Projects/fastai_models/compartment_segmentation/export.pkl'

batch_size = 8
n_cells = len(list_of_test_crops)    

def label_func(x): return x.parent.parent.parent/'annotations'/(x.stem+'.png')
learn = load_learner(segmentation_model_path, cpu=False)

test_dl = learn.dls.test_dl(list_of_test_crops)
preds = torch.argmax(learn.get_preds(dl = test_dl)[0], axis=1)

In [ ]:
ii = 100

for i in range(ii):
    fig = plt.figure(figsize=(8,8))

    ax = fig.add_subplot(1,3,1)
    ax.imshow(list_of_test_crops[i])
    ax.axis('off')
    
    ax = fig.add_subplot(1,3,2)
    ax.imshow(preds[i].numpy())
    ax.axis('off')
    
    ax = fig.add_subplot(1,3,3)
    ax.imshow(list_of_gt_crops[i])
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
item_tfms=Resize((224,224)),
batch_tfms=[*aug_transforms(flip_vert=True), Resize(ResizeMethod.Pad), Normalize.from_stats(*imagenet_stats)]

In [ ]:
jaccard_indices = []
dice_coefficients = []
area_prop = []

for i in range(len(list_of_test_crops)):
    pred_1 = preds[i].numpy() > 0.5
    gt_1 = cv2.resize(list_of_gt_crops[i], (224, 224)) > 0.5 

    intersection = np.logical_and(pred_1, gt_1).sum()
    union = np.logical_or(pred_1, gt_1).sum()
    pred_area = pred_1.sum()
    gt_area = gt_1.sum()

    jaccard = intersection / union if union > 0 else 0
    jaccard_indices.append(jaccard)

    dice = (2 * intersection) / (pred_area + gt_area) if (pred_area + gt_area) > 0 else 0
    dice_coefficients.append(dice)

print(f"Median Jaccard: {np.median(jaccard_indices):.4f}")
print(f"Median Dice: {np.median(dice_coefficients):.4f}")

In [ ]:
data = [jaccard_indices, dice_coefficients]
n = len(jaccard_indices)

print(f"--- Results for {n} Speckle Crops ---")
print(f"Median Jaccard (IoU): {np.median(jaccard_indices):.4f}")
print(f"Median Dice:           {np.median(dice_coefficients):.4f}")

plt.figure(figsize=(4, 5), dpi=300)

bp = plt.boxplot(data, labels=['Jaccard Index\n(IoU)', 'Dice\nCoefficient'], 
                 patch_artist=True, widths=0.6)

colors = ['#1f77b4', '#ff7f0e'] 
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for median in bp['medians']:
    median.set(color='black', linewidth=1)

plt.ylabel('Score', fontsize=14)
plt.title(f'Speckle segmentation (n={n} crops)', fontsize=14, pad=15)
plt.ylim(0.3, 0.95)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()